# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zainabaon/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/zainabaon/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    os.chdir("/content")
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
print(df.shape)

(30000, 45)


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

BLOCKED: the "FlyRank Research: The State of AI-Driven SEO in Numbers" resource is still showing "Lead review pending" and is not yet published on the portal. I've emailed the team to ask for an alternative way to access the paper's findings (draft version, session recording, or notes) and am waiting to hear back. I will complete this section as soon as I have access to real findings from the actual paper — I don't want to fabricate or guess at claims from a document I haven't read, since that would defeat the purpose of this audit exercise.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"].values

# BEFORE: naive random split (wrong — ignores client grouping)
X_train_naive, X_test_naive, y_train_naive, y_test_naive, idx_train, idx_test = train_test_split(
    X, y, df.index, test_size=0.25, random_state=42
)
model_naive = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_train_naive, y_train_naive)
proba_naive = model_naive.predict_proba(X_test_naive)[:,1]
auc_naive = roc_auc_score(y_test_naive, proba_naive)
p50_naive = precision_at_k(proba_naive, y_test_naive, 50)

overlap_naive = set(df.loc[idx_train, "client_id"]) & set(df.loc[idx_test, "client_id"])
print(f"BEFORE (naive random split) — client overlap: {len(overlap_naive)} clients")
print(f"BEFORE — AUC: {auc_naive:.3f}  Precision@50: {p50_naive:.3f}")

# AFTER: client-grouped split (honest, matches Week 5)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
train, test = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

X_train_g = train[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y_train_g = train["is_declining_label"].values
X_test_g = test[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y_test_g = test["is_declining_label"].values

model_grouped = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_train_g, y_train_g)
proba_grouped = model_grouped.predict_proba(X_test_g)[:,1]
auc_grouped = roc_auc_score(y_test_g, proba_grouped)
p50_grouped = precision_at_k(proba_grouped, y_test_g, 50)

overlap_grouped = set(train["client_id"]) & set(test["client_id"])
print(f"\nAFTER (client-grouped split) — client overlap: {len(overlap_grouped)} clients")
print(f"AFTER — AUC: {auc_grouped:.3f}  Precision@50: {p50_grouped:.3f}")

BEFORE (naive random split) — client overlap: 32 clients
BEFORE — AUC: 0.603  Precision@50: 0.640

AFTER (client-grouped split) — client overlap: 0 clients
AFTER — AUC: 0.540  Precision@50: 0.660


BEFORE (naive random split, client overlap = 32): AUC = 0.603, Precision@50 = 0.640.
AFTER (client-grouped split, client overlap = 0): AUC = 0.540, Precision@50 = 0.660.

Mixed result, and worth reporting honestly rather than the expected story: AUC did drop under the honest split (0.603 → 0.540), consistent with the idea that a naive split let the model partially learn client-specific patterns that don't generalize to unseen clients. But Precision@50 actually went UP slightly under the honest split (0.640 → 0.660) — meaning the specific top-50-ranked pages performed better on unseen clients than on the naive split's test set, even though the model's overall discrimination (AUC) got worse.

This shows why checking multiple metrics matters: AUC and Precision@50 measure different things — AUC reflects the model's ranking quality across the whole dataset, while Precision@50 only cares about the very top of the queue, which is what a reviewer actually uses. A single metric alone would have told a misleadingly one-sided story here. The honest split (client overlap = 0, confirmed above) is the trustworthy number to report going forward, and Precision@50 = 0.660 matches what was already reported in Week 5's model comparison, confirming that result was not an artifact of a leaky split.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*
1. Are any features calculated after the decision point? No — all six features are observed same-window snapshots.
2. Does the feature window overlap the target window? Yes, partially — is_declining_label is a same-window proxy (trend_direction == "down"), not a strictly future outcome.
3. Did any product decision flag slip in as a feature? No — health_score, priority_score, action_type are not in this dataset.
4. Does a derived field secretly encode the target? No — trend_pct is deliberately excluded (confirmed in ML-05's leakage hunt).
5. Are duplicate/related rows split across train/test unfairly? This is exactly what Section 2's before/after comparison tests — the naive split allowed client overlap; the grouped split enforces zero overlap.
6. Am I testing on clients the model hasn't effectively seen? Yes, confirmed in the grouped/honest version by the zero-overlap check above.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original (Week 5) claim: "Logistic Regression wins on Precision@50 (0.660 vs baseline 0.560), a real improvement."

Rewritten: "In this observed run, Logistic Regression showed a higher Precision@50 (0.660) than the baseline rule (0.560) on a client-holdout test set. This is a directional, decision-support signal on this specific 30,000-row sample — not a guarantee the same gap holds on new clients or time periods. The same-window proxy label means this measures agreement with a current-state bucket, not a validated future prediction."

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.